# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Show all available record sets with their @id and name
print("Available Record Sets:")
for record_set in metadata.record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}")
    if hasattr(record_set, 'fields'):
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print()

# Example: Print a few records from each record set
for rs in metadata.record_sets:
    print(f"First record(s) from record set '@id={rs.id}' (name={rs.name}):")
    try:
        records = list(dataset.records(record_set=rs.id))
        pprint.pprint(records[:2])
        print()
    except Exception as e:
        print(f"  Could not load records: {e}\n")

## 3. Data Extraction
Load data from specific record set(s) into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Define the list of record_set @ids you want to extract
# We get all record set @ids dynamically
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows for record set: {record_set_id}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Pick the first record set as the main record set for demonstration
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"\nColumns in record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick a numeric field from the main record set
main_df = dataframes[main_record_set_id]

# Attempt to infer a numeric field from available columns
numeric_field = None
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field = col
        break

if numeric_field is None:
    # Try to cast the first non-object column if possible
    for col in main_df.columns:
        try:
            main_df[col] = pd.to_numeric(main_df[col])
            numeric_field = col
            break
        except:
            continue

if numeric_field:
    threshold = main_df[numeric_field].mean() if pd.notnull(main_df[numeric_field].mean()) else 10
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to find a group-able field (string/categorical, not the numeric one)
    group_field = None
    for col in main_df.columns:
        if col != numeric_field and main_df[col].nunique() < len(main_df) / 2:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No numeric field found in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=main_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric or group field for visualization.")

## 6. Conclusion
This notebook demonstrated how to load and explore a Croissant dataset using the `mlcroissant` library.

- The dataset was loaded directly from its Croissant schema URL.
- Available record sets and fields were dynamically listed using their `@id`.
- Data was extracted into pandas DataFrames for convenient analysis.
- Example exploratory data analysis (EDA) and visualizations were provided for numeric fields.

You can continue this notebook by selecting other record sets, deeper analyses, or saving processed data for downstream use.